# Chapter 4 - RQ1 Reconstruction Baseline and Codebook Usage

This notebook is the working notebook for Chapter 4 of the thesis. It assembles the in-distribution ImageNet evidence for the first research question: before perturbing inputs, editing tokens, or changing dataset domains, how do VQGAN and LlamaGen differ in reconstruction quality and codebook usage?

The scope is deliberately narrow. Chapter 4 uses ImageNet as the baseline distribution. ImageNet-V2, ImageNet-Sketch, ObjectNet, BloodMNIST, OrganAMNIST, and RVL-CDIP are useful context for later chapters, but they are not part of the core RQ1 evidence.


## Notebook Contract

This notebook should produce the material that can become Chapter 4 tables, figures, and interpretation notes:

- ImageNet reconstruction metrics for both tokenizers.
- Qualitative reconstruction triplets: original, VQGAN reconstruction, and LlamaGen reconstruction.
- ImageNet codebook usage summaries.
- Usage-distribution and cumulative-mass plots.
- Optional spatial entropy summaries when `position_counts.npy` is available.

The implementation borrows the data layout and helper ideas from `vq-end-to-end-reconstruction-comparison.ipynb` and `vq-code-usage-analysis.ipynb`, but keeps the analysis ImageNet-only and chapter-facing.


## 1. Setup and Reproducibility

The path resolver below assumes the notebook is run from `vq-explore/notebooks`, as in the existing notebooks. It also handles the common case where Jupyter is started from the repository root. Server-side result folders are preferred; local notebook data is used only as a fallback for reconstruction metrics.


In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
})

MODEL_ORDER = ["LlamaGen", "VQGAN"]
MODEL_COLORS = {"LlamaGen": "#2F6F9F", "VQGAN": "#B85C38"}
DATASET_KEY = "imagenet"
DATASET_LABEL = "ImageNet"
CODEBOOK_SIZE = 16_384


def find_notebook_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "notebooks", cwd.parent / "notebooks"]
    for candidate in candidates:
        if (candidate / "data").exists() or candidate.name == "notebooks":
            return candidate
    return cwd


NOTEBOOK_DIR = find_notebook_root()
REPO_ROOT = NOTEBOOK_DIR.parent
THESIS_ROOT = REPO_ROOT.parent / "Thesis---VQ"

END_TO_END_ROOT = REPO_ROOT / "end_to_end_results"
CODE_USAGE_ROOT = REPO_ROOT / "code_usage_results"

LOCAL_RECON_CSV = NOTEBOOK_DIR / "data" / "end_to_end_metrics_numeric.csv"
SERVER_RECON_CSV = END_TO_END_ROOT / "metrics" / "end_to_end_metrics_numeric.csv"

OUTPUT_DIR = NOTEBOOK_DIR / "chapter4_outputs"
DATA_DIR = OUTPUT_DIR / "data"
FIGURE_DIR = OUTPUT_DIR / "figures"
SAMPLE_EXPORT_DIR = OUTPUT_DIR / "reconstruction_samples"

THESIS_ASSET_DIR = THESIS_ROOT / "Assets" / "rq1"
THESIS_DATA_DIR = THESIS_ASSET_DIR / "data"
THESIS_FIGURE_DIR = THESIS_ASSET_DIR / "figures"
THESIS_SAMPLE_DIR = THESIS_ASSET_DIR / "reconstruction_samples"

for directory in [DATA_DIR, FIGURE_DIR, SAMPLE_EXPORT_DIR, THESIS_DATA_DIR, THESIS_FIGURE_DIR, THESIS_SAMPLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Notebook dir:", NOTEBOOK_DIR)
print("Reconstruction CSV:", SERVER_RECON_CSV if SERVER_RECON_CSV.exists() else LOCAL_RECON_CSV)
print("Code usage root:", CODE_USAGE_ROOT)
print("Output dir:", OUTPUT_DIR)


## 2. Loading Helpers

The helper functions are intentionally small. They hide file loading and common summary calculations, while the notebook cells keep the chapter logic visible.


In [ ]:
def save_table(df: pd.DataFrame, name: str) -> Path:
    path = DATA_DIR / name
    df.to_csv(path, index=False)
    thesis_path = THESIS_DATA_DIR / name
    df.to_csv(thesis_path, index=False)
    print(f"saved: {path}")
    print(f"copied: {thesis_path}")
    return path


def save_json(data: dict, name: str) -> Path:
    path = DATA_DIR / name
    path.write_text(json.dumps(data, indent=2) + "\n")
    thesis_path = THESIS_DATA_DIR / name
    thesis_path.write_text(json.dumps(data, indent=2) + "\n")
    print(f"saved: {path}")
    print(f"copied: {thesis_path}")
    return path


def save_figure(fig: plt.Figure, name: str) -> Path:
    path = FIGURE_DIR / name
    fig.savefig(path, bbox_inches="tight")
    thesis_path = THESIS_FIGURE_DIR / name
    fig.savefig(thesis_path, bbox_inches="tight")
    print(f"saved: {path}")
    print(f"copied: {thesis_path}")
    return path


def copy_sample_folder(sample_dir: Path, destination_root: Path) -> Path:
    destination = destination_root / sample_dir.name
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)

    file_map = {
        "orig.png": ["orig.png", "original.png"],
        "vqgan.png": ["vqgan.png"],
        "llamagen.png": ["llamagen.png"],
    }
    for target_name, candidates in file_map.items():
        source = first_existing(sample_dir, candidates)
        if source is None:
            raise FileNotFoundError(f"Missing {target_name} in {sample_dir}")
        shutil.copy2(source, destination / target_name)
    return destination


def require_columns(df: pd.DataFrame, columns: list[str], label: str) -> None:
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def load_reconstruction_metrics() -> pd.DataFrame:
    csv_path = SERVER_RECON_CSV if SERVER_RECON_CSV.exists() else LOCAL_RECON_CSV
    if not csv_path.exists():
        raise FileNotFoundError(
            "No reconstruction metrics CSV found. Expected either "
            f"{SERVER_RECON_CSV} or {LOCAL_RECON_CSV}."
        )
    df = pd.read_csv(csv_path)
    require_columns(df, ["model", "dataset", "psnr_mean", "ssim_mean", "lpips_mean"], "reconstruction metrics")
    return df


def code_usage_dir(model: str, dataset: str = DATASET_KEY) -> Path:
    model_dir = {"LlamaGen": "llamagen", "VQGAN": "vqgan"}[model]
    return CODE_USAGE_ROOT / model_dir / dataset


def load_code_usage_run(model: str, dataset: str = DATASET_KEY) -> dict | None:
    run_dir = code_usage_dir(model, dataset)
    required = ["summary.json", "global_counts.npy", "position_counts.npy", "usage.csv"]
    missing = [name for name in required if not (run_dir / name).exists()]
    if missing:
        print(f"Skipping {model}: missing {missing} in {run_dir}")
        return None
    return {
        "model": model,
        "dataset": dataset,
        "summary": json.loads((run_dir / "summary.json").read_text()),
        "global_counts": np.load(run_dir / "global_counts.npy"),
        "position_counts": np.load(run_dir / "position_counts.npy"),
        "usage": pd.read_csv(run_dir / "usage.csv"),
    }


def token_probabilities(counts: np.ndarray) -> np.ndarray:
    counts = counts.astype(np.float64)
    total = counts.sum()
    if total <= 0:
        return np.zeros_like(counts, dtype=np.float64)
    return counts / total


def entropy_from_probs(probs: np.ndarray) -> float:
    probs = probs[probs > 0]
    return float(-(probs * np.log(probs)).sum())


def count_mass_thresholds(counts: np.ndarray, thresholds=(0.90, 0.95, 0.99, 0.999)) -> dict[str, int]:
    probs = np.sort(token_probabilities(counts))[::-1]
    cumulative = np.cumsum(probs)
    return {f"codes_for_{int(t * 1000) / 10:g}pct_mass": int(np.searchsorted(cumulative, t) + 1) for t in thresholds}


def effective_active_count(counts: np.ndarray, min_fraction: float = 1e-6) -> int:
    total = counts.sum()
    if total <= 0:
        return 0
    return int((counts / total >= min_fraction).sum())


def summarize_counts(model: str, run: dict, min_fraction: float = 1e-6) -> dict:
    counts = run["global_counts"]
    probs = token_probabilities(counts)
    entropy = entropy_from_probs(probs)
    summary = run["summary"]
    row = {
        "model": model,
        "dataset": DATASET_LABEL,
        "n_images": summary.get("n_images"),
        "token_grid_hw": str(summary.get("token_grid_hw")),
        "total_tokens": int(counts.sum()),
        "strict_active_codes": int((counts > 0).sum()),
        "effective_active_codes_1e-6": effective_active_count(counts, min_fraction),
        "dead_codes": int((counts == 0).sum()),
        "active_fraction_full": float((counts > 0).sum() / len(counts)),
        "entropy_nats": entropy,
        "perplexity": float(np.exp(entropy)),
        "top_10_mass": float(np.sort(probs)[::-1][:10].sum()),
        "top_100_mass": float(np.sort(probs)[::-1][:100].sum()),
        "top_500_mass": float(np.sort(probs)[::-1][:500].sum()),
    }
    row.update(count_mass_thresholds(counts))
    return row


def position_entropy(position_counts: np.ndarray) -> np.ndarray:
    totals = position_counts.sum(axis=0, keepdims=True)
    probs = np.divide(
        position_counts,
        totals,
        out=np.zeros_like(position_counts, dtype=np.float64),
        where=totals > 0,
    )
    log_probs = np.zeros_like(probs)
    mask = probs > 0
    log_probs[mask] = np.log(probs[mask])
    return -(probs * log_probs).sum(axis=0)


## 3. Reconstruction Baseline

This section isolates the ImageNet rows from the end-to-end reconstruction table. These metrics establish the clean-image baseline for the rest of the thesis.


In [ ]:
recon_all = load_reconstruction_metrics()
save_table(recon_all, "end_to_end_metrics_numeric.csv")

recon_imagenet = (
    recon_all[recon_all["dataset"].eq(DATASET_LABEL) & recon_all["model"].isin(MODEL_ORDER)]
    .copy()
    .set_index("model")
    .loc[MODEL_ORDER]
    .reset_index()
)

recon_columns = [
    "model", "dataset", "psnr_mean", "psnr_std", "ssim_mean", "ssim_std",
    "lpips_mean", "lpips_std", "fid", "sfid", "precision", "recall", "inception",
]
recon_table = recon_imagenet[recon_columns].copy()
save_table(recon_table, "ch04_table_reconstruction_metrics_imagenet.csv")
display(recon_table)


### Reconstruction Reading

For Chapter 4, the key comparison is not only whether one tokenizer is better on a single number. PSNR and SSIM measure aligned fidelity, LPIPS measures learned perceptual distance, and FID/sFID/precision/recall describe distributional behavior. The written chapter should present these as complementary views of the baseline reconstruction problem.


In [ ]:
metric_specs = [
    ("psnr_mean", "psnr_std", "PSNR", "higher"),
    ("ssim_mean", "ssim_std", "SSIM", "higher"),
    ("lpips_mean", "lpips_std", "LPIPS", "lower"),
    ("fid", None, "FID", "lower"),
    ("sfid", None, "sFID", "lower"),
    ("precision", None, "Precision", "higher"),
    ("recall", None, "Recall", "higher"),
    ("inception", None, "Inception Score", "higher"),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 6.5), constrained_layout=True)
axes = axes.ravel()

for ax, (metric, std_metric, label, direction) in zip(axes, metric_specs):
    values = recon_imagenet.set_index("model").loc[MODEL_ORDER, metric]
    errors = recon_imagenet.set_index("model").loc[MODEL_ORDER, std_metric] if std_metric else None
    colors = [MODEL_COLORS[model] for model in MODEL_ORDER]
    ax.bar(MODEL_ORDER, values, yerr=errors, capsize=3 if std_metric else 0, color=colors, alpha=0.9)
    ax.set_title(f"{label}\n({direction} is better)", fontsize=10)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y")

fig.suptitle("ImageNet reconstruction baseline", fontsize=14, fontweight="bold")
save_figure(fig, "ch04_fig_reconstruction_metric_comparison.png")
plt.show()


## 4. Qualitative Reconstruction Samples

The old reconstruction notebook uses side-by-side folders under `end_to_end_results/side_by_side`. The helper below accepts both `original.png` and `orig.png`, since both naming conventions have appeared in notes or generated artifacts.


In [ ]:
SIDE_BY_SIDE_ROOT = END_TO_END_ROOT / "side_by_side" / DATASET_KEY


def sample_dirs(limit: int | None = None) -> list[Path]:
    if not SIDE_BY_SIDE_ROOT.exists():
        print(f"No qualitative sample directory found: {SIDE_BY_SIDE_ROOT}")
        return []
    dirs = sorted(p for p in SIDE_BY_SIDE_ROOT.iterdir() if p.is_dir() and p.name.startswith("sample_"))
    return dirs if limit is None else dirs[:limit]


def first_existing(sample_dir: Path, names: list[str]) -> Path | None:
    for name in names:
        path = sample_dir / name
        if path.exists():
            return path
    return None


def reconstruction_triplet_paths(sample_dir: Path) -> dict[str, Path] | None:
    paths = {
        "Original": first_existing(sample_dir, ["original.png", "orig.png"]),
        "VQGAN": first_existing(sample_dir, ["vqgan.png"]),
        "LlamaGen": first_existing(sample_dir, ["llamagen.png"]),
    }
    if any(path is None for path in paths.values()):
        print(f"Skipping incomplete sample: {sample_dir}")
        return None
    return paths


def export_reconstruction_sample_folders(sample_count: int = 6) -> list[Path]:
    exported = []
    for sample_dir in sample_dirs(sample_count):
        if reconstruction_triplet_paths(sample_dir) is None:
            continue
        exported.append(copy_sample_folder(sample_dir, SAMPLE_EXPORT_DIR))
        copy_sample_folder(sample_dir, THESIS_SAMPLE_DIR)
    print(f"exported {len(exported)} sample folders")
    return exported


def plot_reconstruction_triplets(sample_count: int = 4) -> plt.Figure | None:
    dirs = sample_dirs(sample_count)
    triplets = [(d, reconstruction_triplet_paths(d)) for d in dirs]
    triplets = [(d, t) for d, t in triplets if t is not None]
    if not triplets:
        print("No complete reconstruction triplets available.")
        return None

    fig, axes = plt.subplots(len(triplets), 3, figsize=(9, 3 * len(triplets)), squeeze=False)
    for row, (sample_dir, paths) in enumerate(triplets):
        for col, (title, path) in enumerate(paths.items()):
            ax = axes[row, col]
            ax.imshow(Image.open(path).convert("RGB"))
            ax.set_title(title if row == 0 else "")
            ax.set_ylabel(sample_dir.name, fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)
    fig.suptitle("Qualitative ImageNet reconstruction examples", fontsize=14, fontweight="bold")
    return fig


exported_samples = export_reconstruction_sample_folders(sample_count=6)
triplet_fig = plot_reconstruction_triplets(sample_count=min(4, max(1, len(exported_samples))))
if triplet_fig is not None:
    save_figure(triplet_fig, "ch04_fig_reconstruction_triplets.png")
    plt.show()


## 5. Codebook Usage

This section loads ImageNet code-usage outputs. The strict active-code count is reported for continuity with the older notebook, but the chapter should also report effective usage measures such as entropy, perplexity, cumulative mass, and thresholded active counts.


In [ ]:
runs = {model: load_code_usage_run(model) for model in MODEL_ORDER}
runs = {model: run for model, run in runs.items() if run is not None}
CODEBOOK_AVAILABLE = set(runs) == set(MODEL_ORDER)

if CODEBOOK_AVAILABLE:
    codebook_summary = pd.DataFrame([summarize_counts(model, runs[model]) for model in MODEL_ORDER])
    rq1_summary = recon_table.merge(codebook_summary, on="model", suffixes=("_recon", "_codebook"))

    save_table(codebook_summary, "ch04_table_codebook_usage_imagenet.csv")
    save_table(rq1_summary, "rq1_imagenet_reconstruction_codebook_summary.csv")

    for model in MODEL_ORDER:
        slug = model.lower()
        usage = runs[model]["usage"].copy()
        if "count" in usage.columns:
            usage = usage.sort_values("count", ascending=False)
        save_table(usage, f"rq1_imagenet_{slug}_code_usage.csv")
        save_json(runs[model]["summary"], f"{slug}_imagenet_summary.json")

    display(rq1_summary)
else:
    print("Codebook usage results are incomplete. Reconstruction sections remain runnable; codebook figures are skipped.")


### Codebook Validation

The count arrays should contain exactly one assignment per image token. For the 16-by-16 tokenizers, the expected token count is `n_images * 256`. This validation catches broken or partial exports before the values are used in the thesis.


In [ ]:
if CODEBOOK_AVAILABLE:
    validation_rows = []
    for model, run in runs.items():
        summary = run["summary"]
        h, w = summary.get("token_grid_hw", [16, 16])
        expected = int(summary["n_images"] * h * w)
        validation_rows.append({
            "model": model,
            "global_total": int(run["global_counts"].sum()),
            "position_total": int(run["position_counts"].sum()),
            "expected_total": expected,
            "ok": int(run["global_counts"].sum()) == int(run["position_counts"].sum()) == expected,
        })
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)
    if not validation_df["ok"].all():
        raise ValueError("Codebook count validation failed. Inspect exports before using these results.")
else:
    print("Skipped because codebook usage results are incomplete.")


### Sorted Usage Distribution

A sorted usage curve shows concentration directly: the x-axis is token rank after sorting by frequency, and the y-axis is the fraction of all token assignments taken by that rank. A steep curve means that a small subset of codes carries a large share of all tokens.


In [ ]:
if CODEBOOK_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for model in MODEL_ORDER:
        counts = runs[model]["global_counts"]
        freqs = np.sort(token_probabilities(counts))[::-1]
        ax.plot(np.arange(1, len(freqs) + 1), freqs, label=model, color=MODEL_COLORS[model], linewidth=2)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Code rank by frequency")
    ax.set_ylabel("Token-assignment frequency")
    ax.set_title("ImageNet sorted code usage")
    ax.legend(frameon=False)
    ax.grid(True, which="both")
    save_figure(fig, "ch04_fig_code_usage_sorted.png")
    plt.show()
else:
    print("Skipped because codebook usage results are incomplete.")


### Cumulative Mass

Cumulative mass complements the log-log sorted curve. It answers a thesis-friendly question: how many codes are needed to explain a given fraction of all token assignments?


In [ ]:
if CODEBOOK_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for model in MODEL_ORDER:
        freqs = np.sort(token_probabilities(runs[model]["global_counts"]))[::-1]
        cumulative = np.cumsum(freqs)
        ax.plot(np.arange(1, len(cumulative) + 1), cumulative, label=model, color=MODEL_COLORS[model], linewidth=2)

    for threshold in [0.90, 0.95, 0.99]:
        ax.axhline(threshold, color="0.35", linestyle="--", linewidth=0.9)
        ax.text(CODEBOOK_SIZE * 1.02, threshold, f"{threshold:.0%}", va="center", fontsize=8)

    ax.set_xscale("log")
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Number of most frequent codes included")
    ax.set_ylabel("Cumulative token mass")
    ax.set_title("ImageNet cumulative code-usage mass")
    ax.legend(frameon=False)
    ax.grid(True, which="both")
    save_figure(fig, "ch04_fig_code_usage_cumulative_mass.png")
    plt.show()
else:
    print("Skipped because codebook usage results are incomplete.")


## 6. Spatial Codebook Usage

Global frequency alone does not show whether a tokenizer uses codes uniformly across the 16-by-16 grid or whether some positions have lower effective vocabulary diversity. The positional entropy heatmap computes the entropy of code assignments at each token-grid location.


In [ ]:
if CODEBOOK_AVAILABLE:
    entropy_grids = {model: position_entropy(runs[model]["position_counts"]) for model in MODEL_ORDER}
    vmin = min(grid.min() for grid in entropy_grids.values())
    vmax = max(grid.max() for grid in entropy_grids.values())

    fig, axes = plt.subplots(1, 2, figsize=(8.5, 4), constrained_layout=True)
    for ax, model in zip(axes, MODEL_ORDER):
        im = ax.imshow(entropy_grids[model], cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_title(model)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04, label="Entropy (nats)")
    fig.suptitle("Per-position ImageNet code entropy", fontsize=14, fontweight="bold")
    save_figure(fig, "ch04_fig_position_entropy.png")
    plt.show()
else:
    print("Skipped because codebook usage results are incomplete.")


## 7. Joint Reconstruction and Codebook View

This table joins the two Chapter 4 evidence streams. It is the best compact view for the thesis discussion because it places reconstruction fidelity next to vocabulary utilization.


In [ ]:
if CODEBOOK_AVAILABLE:
    joint_table = recon_table.merge(codebook_summary, on="model", suffixes=("_recon", "_codebook"))
    joint_table = joint_table[[
        "model", "psnr_mean", "ssim_mean", "lpips_mean", "fid", "sfid",
        "strict_active_codes", "effective_active_codes_1e-6", "perplexity",
        "top_10_mass", "top_100_mass", "codes_for_90pct_mass", "codes_for_99pct_mass",
    ]]
    save_table(joint_table, "ch04_table_joint_reconstruction_codebook_imagenet.csv")
    display(joint_table)
else:
    print("Skipped because codebook usage results are incomplete.")


## 8. Thesis Notes Draft

Use this cell as a lightweight checklist while writing Chapter 4. Keep claims tied to the exported tables and figures.

- Reconstruction baseline: compare the models across pixel, perceptual, and distributional metric families rather than from a single scalar.
- Codebook usage: distinguish strict active-code count from effective vocabulary use measured by entropy, perplexity, and cumulative mass.
- Joint interpretation: ask whether stronger reconstruction coincides with broader or more balanced codebook use.
- Boundary of the chapter: do not explain robustness or distribution-shift behavior here; use Chapter 4 to establish the baseline that those later chapters perturb.


## 9. Artifact Checklist

After a clean server run, the notebook should create one Chapter 4 output bundle under `chapter4_outputs/` and copy the same files to `Thesis---VQ/Assets/rq1/`.

Expected data files:

- `data/end_to_end_metrics_numeric.csv`
- `data/rq1_imagenet_reconstruction_codebook_summary.csv`
- `data/rq1_imagenet_llamagen_code_usage.csv`
- `data/rq1_imagenet_vqgan_code_usage.csv`
- `data/llamagen_imagenet_summary.json`
- `data/vqgan_imagenet_summary.json`
- `data/ch04_table_reconstruction_metrics_imagenet.csv`
- `data/ch04_table_codebook_usage_imagenet.csv`
- `data/ch04_table_joint_reconstruction_codebook_imagenet.csv`

Expected reconstruction images:

- `reconstruction_samples/sample_*/orig.png`
- `reconstruction_samples/sample_*/vqgan.png`
- `reconstruction_samples/sample_*/llamagen.png`
- `figures/ch04_fig_reconstruction_triplets.png`

Expected code-usage figures:

- `figures/ch04_fig_code_usage_sorted.png`
- `figures/ch04_fig_code_usage_cumulative_mass.png`
- `figures/ch04_fig_position_entropy.png`

The codebook outputs require `../code_usage_results/{llamagen,vqgan}/imagenet/`. The sample-folder outputs require `../end_to_end_results/side_by_side/imagenet/sample_*/`.
